In [10]:
# config.py

#--------Packages
from pathlib import Path

### Below from SMM notes
import pandas as pd
import matplotlib.pyplot as plt
import dask
from dask import delayed, compute
from dask.distributed import Client
import scipy.optimize as optimize
from scipy.optimize import minimize
import scipy.stats as stats
import plotly.express as px
import numpy as np
import sys
from functools import lru_cache
import warnings
import numpy as np
import pandas as pd
from scipy.optimize import minimize


!{sys.executable} -m pip install quantecon
import quantecon as qe
from quantecon.markov.approximation import tauchen

#---------relative path configuration

# Root = folder containing config.py

from pathlib import Path

try:
    ROOT = Path(__file__).resolve().parent.parent
except NameError:
    ROOT = Path.cwd()

SCRIPTS = ROOT / "scripts"
RESULTS = ROOT / "results"

RESULTS.mkdir(exist_ok=True)

print("Setup successful")
print(f"ROOT: {ROOT}")
print(f"SCRIPTS: {SCRIPTS}")
print(f"RESULTS: {RESULTS}")



Setup successful
ROOT: /Users/marlifirillo/CompEcon_Fall25/DynamicProgramming/ProblemSet8
SCRIPTS: /Users/marlifirillo/CompEcon_Fall25/DynamicProgramming/ProblemSet8/scripts
RESULTS: /Users/marlifirillo/CompEcon_Fall25/DynamicProgramming/ProblemSet8/results


In [11]:
#Module 1: Define the model parameters and set up targets moments from Table 3.

#--------CE Calibrated Parameters

beta = 0.95      # discount factor
delta = 0.15     # depreciation rate
p = 1.0          # price of capital

#--------SMM starting parameters

alpha = 0.6956
gamma = 0.1331
rho = 0.0976
sigma = 0.8932
phi_tilde0 = 0.0


#Psi d - AKA target-moments

target_moments = [
    0.03,   # a1 - coefficient on expected average Q
    0.24,   # a2 - coefficient on the profit rate
    0.40,   # serial correlation of investment rate
    0.25,   # std of profit rate (pi/K)
    3.00,   # average Q
    0.25    # fraction externally financed
]


# Parameter names used in SMM estimation
PARAMETER_NAMES = [
    "alpha",
    "gamma",
    "rho",
    "sigma",
    "phi_tilde0"
]

# Six moments matched between simulated and actual data
MOMENT_NAMES = [
    "a1",
    "a2",
    "sc(I/K)",
    "std(CF/K)",
    "mean Q",
    "external fraction"
]

#target_moments = np.asarray(target_moments, dtype=float)

# weighting_matrix = np.eye(len(target_moments))  # Weighting matrix will be an identity matrix for simplicity



In [12]:
# Module 3: Define functions for use in later steps

# 1. Profit function 
def profit(K, A, alpha):
    """
    Calculate firm profit given capital K and productivity A.
    """
    return A * K**alpha

# 2. Investment function 
def investment(K, K_next, delta):
    """
    Calculate gross investment needed to move from K to K_next.
    """
    return K_next - (1 - delta) * K

# 3. Adjustment costs now use GROSS investment from Equation (4)
def adjustment_costs(K, K_next, delta, gamma):
    """
    Calculate adjustment costs using gross investment from Equation (4).
    """
    I = investment(K, K_next, delta)
    return (gamma / 2) * (I / K) ** 2 * K


# Calculate current period net cash flow
def current_payout(K, A, alpha, K_next, delta, gamma):
    """
    Calculate current period net cash flow after investment and adjustment costs.
    """
    return (
        profit(K, A, alpha)
        - investment(K, K_next, delta)
        - adjustment_costs(K, K_next, delta, gamma)
    )

In [13]:
# Module 4: Settings for simulation and standard error exercise
# Settings used for simulation and inference
INFERENCE_CONFIG = dict(
    K_min=1.0,
    K_max=1000.0,
    nK=100,
    nA=5,
    num_firms=500,
    num_periods=100,
    burn_in=30,
    delta=float(delta),
    beta=float(beta),
    phi1=0.0
)

# Number of simulated panels used for each SMM objective
S_ESTIMATION = 1

# Number of independent panels used to estimate moment covariance
R_COVARIANCE = 100

# Number of panels used to reduce noise in numerical derivatives
S_DERIVATIVE = 10

# Seeds make simulated results reproducible
ESTIMATION_SEEDS = tuple(
    12345 + s for s in range(S_ESTIMATION)
)

DERIVATIVE_SEEDS = tuple(
    54321 + s for s in range(S_DERIVATIVE)
)

COVARIANCE_SEEDS = tuple(
    90000 + r for r in range(R_COVARIANCE)
)

# Bounds for each parameter during estimation
INFERENCE_BOUNDS = np.array([
    (0.40, 0.95),
    (0.001, 2.0),
    (0.001, 0.98),
    (0.05, 1.5),
    (0.0, 2.0)
])

# Step sizes used to calculate numerical derivatives
FD_STEPS = np.array([
    0.005,
    0.01,
    0.01,
    0.02,
    0.02
])

# Maximum number of times the optimizer can evaluate the objective
MAX_FUNCTION_EVALUATIONS = 1000

#Define grids


def make_A_process(rho, sigma, nA=5):
    # QuantEcon tauchen method wrapper
    mc = tauchen(n=nA, rho=rho, sigma=sigma)
    logA_grid = np.asarray(mc.state_values)
    P = np.asarray(mc.P)

    # Ensure transition probabilities are valid
    P = np.where(P < 0, 0.0, P)
    P = P / P.sum(axis=1, keepdims=True)
    A_grid = np.exp(logA_grid)

    return A_grid, P

def make_K_grid(K_min=1.0, K_max=1000.0, nK=300):
    return np.linspace(K_min, K_max, nK)




In [14]:
# Module 5: Solve DP model, simulate firms  

def solve_inference_dp(theta, cfg):
    """
    Solve the model for a given set of parameters using value function iteration.

    Returns the grids, transition matrix, value function, capital policy,
    and financing policy used to simulate the model.
    """
    alpha, gamma, rho, sigma, phi_tilde0 = np.asarray(theta, dtype=float)

    # Create productivity process and capital grid
    A_grid, P = make_A_process(rho, sigma, nA=cfg["nA"])
    K_grid = make_K_grid(
        cfg["K_min"],
        cfg["K_max"],
        nK=cfg["nK"]
    )

    # Reshape grids to evaluate all combinations of K, K_next, and A
    K = K_grid[:, None, None]
    K_next = K_grid[None, :, None]
    A = A_grid[None, None, :]

    # Calculate profit and investment for each possible choice
    pi = profit(K, A, alpha)
    I = investment(K, K_next, cfg["delta"])

    # Determine whether external financing is needed
    external_finance = np.maximum(I - pi, 0.0)
    external = I > pi

    # Calculate payout before external financing costs
    net_funds = (
        pi
        - I
        - adjustment_costs(
            K,
            K_next,
            cfg["delta"],
            gamma
        )
    )

    # Apply external financing costs when external financing is needed
    equity_issuance_cost = (
        phi_tilde0
        + cfg["phi1"] * external_finance
    )

    payout = net_funds - np.where(
        external,
        equity_issuance_cost,
        0.0
    )

    # Initialize value function
    V = np.zeros((len(K_grid), len(A_grid)))

    # Value function iteration
    for iteration in range(1, 1001):

        # Expected value for each possible choice of K_next
        expected_value = V @ P.T

        # Bellman equation
        rhs = (
            payout
            + cfg["beta"] * expected_value[None, :, :]
        )

        # Find optimal K_next for each K and A
        policy = np.argmax(rhs, axis=1)

        # Store value associated with optimal choice
        V_new = np.take_along_axis(
            rhs,
            policy[:, None, :],
            axis=1
        )[:, 0, :]

        # Check convergence
        V_dist = np.max(np.abs(V_new - V))
        V = V_new

        if V_dist < 1e-6:
            break

    else:
        raise ValueError(
            "Inference DP did not converge."
        )

    # Record whether optimal choice uses external financing
    finance_policy = np.take_along_axis(
        external,
        policy[:, None, :],
        axis=1
    )[:, 0, :].astype(int)

    return K_grid, A_grid, P, V, policy, finance_policy

#################### Part2 ####################

def simulate_panel(theta, solved, seed, cfg):
    """
    Simulate a panel of firms using the solved policy function.

    Generates investment, cash flow, Q, expected Q, and financing data
    for each firm and period after the burn-in period.
    """
    alpha, gamma, rho, sigma, phi_tilde0 = np.asarray(theta, dtype=float)

    K_grid, A_grid, P, V, policy, finance_policy = solved

    num_firms = cfg["num_firms"]
    num_periods = cfg["num_periods"]
    burn_in = cfg["burn_in"]

    if num_firms < 2 or not 0 <= burn_in < num_periods - 1:
        raise ValueError(
            "Need at least two firms and two retained periods."
        )

    # Generate random draws used for productivity transitions
    rng = np.random.default_rng(seed)
    uniforms = rng.random((num_firms, num_periods))

    # Start all firms in the middle of the K and A grids
    K_index = np.full(
        num_firms,
        len(K_grid) // 2,
        dtype=int
    )

    A_index = np.full(
        num_firms,
        len(A_grid) // 2,
        dtype=int
    )

    retained_periods = num_periods - burn_in

    # Store simulated firm data
    investment_rate = np.empty(
        (num_firms, retained_periods)
    )

    cash_flow = np.empty(
        (num_firms, retained_periods)
    )

    expected_Q = np.empty(
        (num_firms, retained_periods)
    )

    Q = np.empty(
        (num_firms, retained_periods)
    )

    financing = np.empty(
        (num_firms, retained_periods)
    )

    gross_investment = np.empty((num_firms, retained_periods))
    external_finance = np.empty((num_firms, retained_periods))

    # Cumulative probabilities used to draw next-period productivity
    cumulative_P = np.cumsum(P, axis=1)
    cumulative_P[:, -1] = 1.0

    # Simulate firms over time
    for t in range(num_periods):

        # Choose next-period capital using the policy function
        K_next_index = policy[K_index, A_index]

        K = K_grid[K_index]
        K_next = K_grid[K_next_index]
        A = A_grid[A_index]

        # Keep observations after burn-in
        if t >= burn_in:
            j = t - burn_in

            # Gross investment and the amount financed externally
            I = investment(K, K_next, cfg["delta"])
            gross_investment[:, j] = I
            external_finance[:, j] = np.maximum(
                I - profit(K, A, alpha), 0.0
            )

            # Investment divided by capital
            investment_rate[:, j] = (
                investment(
                    K,
                    K_next,
                    cfg["delta"]
                ) / K
            )

            # Cash flow divided by capital
            cash_flow[:, j] = (
                profit(K, A, alpha) / K
            )

            # Expected Q used in the investment regression
            expected_Q[:, j] = (
                np.sum(
                    P[A_index, :] * V[K_next_index, :],
                    axis=1
                ) / K_next
            )

            # Current Q
            Q[:, j] = (
                V[K_index, A_index] / K
            )

            # External financing decision
            financing[:, j] = (
                finance_policy[K_index, A_index]
            )

        # Move capital to next period
        K_index = K_next_index

        # Draw next-period productivity
        A_index = np.sum(
            uniforms[:, t, None]
            > cumulative_P[A_index, :],
            axis=1
        )

    return {
        "investment_rate": investment_rate,
        "cash_flow": cash_flow,
        "expected_Q": expected_Q,
        "Q": Q,
        "financing": financing,
        "gross_investment": gross_investment,
        "external_finance": external_finance
    }




In [15]:
#Module 6:

def calculate_moments(panel):
    """
    Calculate the six moments used for SMM from the simulated panel.

    Uses the simulated investment, cash flow, Q, and financing data
    to calculate the moments matched to the actual data.
    """
    investment_rate = panel["investment_rate"]
    cash_flow = panel["cash_flow"]
    expected_Q = panel["expected_Q"]
    Q = panel["Q"]
    gross_investment = panel["gross_investment"]
    external_finance = panel["external_finance"]
    total_investment = np.sum(gross_investment)
    if total_investment <= 0:
        raise ValueError("External financing fraction requires positive total investment.")
    external_fraction = np.sum(external_finance) / total_investment

    # Regression of investment on expected Q and cash flow
    X = np.column_stack((
        np.ones(investment_rate.size),
        expected_Q.ravel(),
        cash_flow.ravel()
    ))

    if np.linalg.matrix_rank(X) < 3:
        raise ValueError(
            "Simulated investment regression is rank deficient."
        )

    regression_results = np.linalg.lstsq(
        X,
        investment_rate.ravel(),
        rcond=None
    )[0]

    _, a1, a2 = regression_results

    # Calculate serial correlation of investment
    investment_lag = investment_rate[:, :-1].ravel()
    investment_lead = investment_rate[:, 1:].ravel()

    if min(
        np.std(investment_lag),
        np.std(investment_lead)
    ) < 1e-12:
        raise ValueError(
            "Investment serial correlation is undefined at this parameter trial."
        )

    # Calculate six simulated moments
    moments = np.array([
        a1,
        a2,
        np.corrcoef(
            investment_lag,
            investment_lead
        )[0, 1],
        np.std(cash_flow),
        np.mean(Q),
        external_fraction
    ])

    if not np.isfinite(moments).all():
        raise ValueError(
            "Non-finite simulated moments."
        )

    return moments


def inference_panel_moments(theta, solved, seed, cfg):
    """
    Simulate one panel and calculate its six moments.
    """
    panel = simulate_panel(
        theta,
        solved,
        seed,
        cfg
    )

    return calculate_moments(panel)


def inference_mean_moments(theta, seeds, cfg=INFERENCE_CONFIG):
    """
    Calculate average simulated moments across panels using different seeds.

    The model is solved once for each parameter set and the resulting
    policy function is used for each simulated panel.
    """
    solved = solve_inference_dp(theta, cfg)

    panel_moments = [
        inference_panel_moments(
            theta,
            solved,
            seed,
            cfg
        )
        for seed in seeds
    ]

    return np.mean(panel_moments, axis=0)

In [16]:
# Module 7: SMM optimization and second-stage weighting matrix

def estimate_inference_stage(start, W, label, maxfev=MAX_FUNCTION_EVALUATIONS):
    """
    Estimate the five structural parameters for one fixed weighting matrix.

    Each parameter trial solves the model, simulates panels using fixed seeds,
    and minimizes the weighted discrepancy between simulated and target moments.
    Uses Nelder Mead to find best evaluated trial.

    Parameters
    ----------
    start : array-like, shape (5,)
        Initial values ordered as alpha, gamma, rho, sigma, phi_tilde0.
    W : array-like, shape (6, 6)
        Weighting matrix, held fixed throughout this estimation stage.
    label : str
        Stage name printed in progress messages.
    maxfev : int
        Evaluation limit for EACH optimizer, rather than the combined total.

    Returns
    -------
    dict
        theta: best evaluated parameters; loss: their unscaled SMM criterion;
        optimizer: Nelder-Mead result; exploration: Powell result;
        converged: whether refinement succeeded and ended at the best loss;
        history: unscaled losses from newly evaluated parameter vectors.

    Notes
    -----
    Exact repeated trials reuse cached losses. Trials that raise ValueError
    receive a penalty of 1e12. Objective scaling changes the numerical scale
    but does not change which parameters minimize the criterion.
    """
    W = np.asarray(W, dtype=float)
    if W.shape != (6, 6) or not np.isfinite(W).all():
        raise ValueError("W must be a finite 6-by-6 matrix.")
    history, cache = [], {}
    best = {"theta": np.array(start, dtype=float), "loss": np.inf}
    def raw_objective(theta):
        """Return the unscaled SMM loss and track newly evaluated trials."""
        key = tuple(np.asarray(theta, dtype=float))
        if key in cache:
            return cache[key]
        try:
            m = inference_mean_moments(theta, ESTIMATION_SEEDS)
            gap = m - np.asarray(target_moments)
            loss = float(gap @ W @ gap)
        except ValueError:
            loss = 1e12
        cache[key] = loss
        if loss < best["loss"]:
            best.update(theta=np.array(theta).copy(), loss=loss)
        history.append(loss)
        if len(history) == 1 or len(history) % 25 == 0:
            print(f"{label}: evaluation {len(history)}, best loss {best['loss']:.6f}", flush=True)
        return loss
    # Scale the criterion using the initial loss for numerical optimization.
    scale = max(raw_objective(start), 1e-8)
    objective = lambda theta: raw_objective(theta) / scale
    # Step 1: bounded Nelder-Mead search.
    exploration = minimize(objective, np.asarray(start, dtype=float), method="Nelder-Mead",
                           bounds=INFERENCE_BOUNDS,
                           options=dict(maxfev=maxfev, xtol=1e-3, ftol=1e-3))
    # Grid-induced discontinuities can cause Nelder-Mead to return a worse terminal point.
    # Refine the best evaluated point and explicitly check the terminal result.
    simplex = np.tile(best["theta"], (6, 1))
    for j in range(5):
        h = FD_STEPS[j]
        direction = 1 if best["theta"][j] + h <= INFERENCE_BOUNDS[j, 1] else -1
        simplex[j+1, j] += direction * h
    # Step 2: bounded simplex refinement, starting at the best trial.
    fit = minimize(objective, best["theta"], method="Nelder-Mead", bounds=INFERENCE_BOUNDS,
                   options=dict(maxfev=maxfev, initial_simplex=simplex,
                                xatol=1e-4, fatol=1e-6))
    terminal_loss = raw_objective(fit.x)
    terminal_is_best = abs(terminal_loss - best["loss"]) <= 1e-6 * max(1.0, best["loss"])
    converged = bool(fit.success and terminal_is_best)
    print(f"{label}: converged={converged}; {fit.message}", flush=True)
    if best["loss"] >= 1e12:
        raise ValueError("No valid parameter trial found.")
    return dict(theta=best["theta"], loss=best["loss"], optimizer=fit,
                exploration=exploration, converged=converged, history=history)


def estimate_moment_covariance(theta, seeds=COVARIANCE_SEEDS, cfg=INFERENCE_CONFIG):
    """
    Estimate panel-moment covariance and the second-stage weighting matrix.

    Solve the model once at the supplied parameters, then simulate independent
    panels. Each panel contributes one row containing its six moments.

    Parameters
    ----------
    theta : array-like, shape (5,)
        Parameters at which covariance is evaluated, typically Stage 1 estimates.
    seeds : sequence of int
        More than six distinct seeds for independent simulated panels.
    cfg : dict
        Model-grid and simulation settings.

    Returns
    -------
    rows : ndarray, shape (R, 6)
        Moment vectors from R simulated panels.
    Omega : ndarray, shape (6, 6)
        Sample covariance of one panel's moment vector, using ddof=1.
    W_star : ndarray, shape (6, 6)
        Symmetrized inverse of Omega, used as the Stage 2 weighting matrix.

    Raises
    ------
    ValueError
        If seeds are insufficient or repeated, or covariance is singular or
        too ill conditioned to invert reliably.

    Notes
    -----
    R is the number of panels used to estimate covariance. It is distinct from
    S, the number of panels averaged in the estimation objective. Omega is not
    divided by R: it describes variability across individual panel moments.
    """
    if len(seeds) <= 6 or len(set(seeds)) != len(seeds):
        raise ValueError("Use more than six distinct independent panel seeds.")
    solved = solve_inference_dp(theta, cfg)
    rows = []
    for i, seed in enumerate(seeds, start=1):
        rows.append(inference_panel_moments(theta, solved, seed, cfg))
        if i % 25 == 0:
            print(f"Covariance panels: {i}/{len(seeds)}", flush=True)
    rows = np.asarray(rows)
    Omega = np.cov(rows, rowvar=False, ddof=1)
    # A pseudoinverse can manufacture misleading precision; require invertibility.
    eigenvalues = np.linalg.eigvalsh(Omega)
    if eigenvalues[0] <= 0 or np.linalg.cond(Omega) > 1e12:
        raise ValueError("Moment covariance is singular or ill conditioned; increase panels or diagnose redundant moments.")
    W_star = np.linalg.solve(Omega, np.eye(6))
    return rows, Omega, (W_star + W_star.T) / 2

In [17]:
# Module 8: Numerical moment derivatives and parameter standard errors

def numerical_moment_jacobian(theta, steps=FD_STEPS, seeds=DERIVATIVE_SEEDS):
    """
    Approximate how the six simulated moments respond to five parameters.

    Parameters
    ----------
    theta : array-like, shape (5,)
        Parameters at which derivatives are evaluated, usually Stage 2 estimates.
    steps : array-like, shape (5,)
        Positive perturbation size for each parameter.
    seeds : sequence of int
        Same simulation seeds used at the base and perturbed parameters to
        reduce simulation noise in the differences.

    Returns
    -------
    D : ndarray, shape (6, 5)
        Moment Jacobian: D[i, j] approximates the derivative of moment i
        with respect to parameter j.
    methods : list of str
        Difference method for each parameter: central, forward, or backward.

    Notes
    -----
    Central differences use trials above and below the estimate. At parameter
    bounds, a one-sided difference is used. Each trial solves and simulates
    the model again. These are derivatives of moments, not of the SMM loss.
    """
    theta = np.asarray(theta, dtype=float)
    # Moments at the unperturbed estimate, used for one-sided differences.
    base = inference_mean_moments(theta, seeds)
    D = np.empty((6, 5))
    methods = []
    # Build one column of D for each structural parameter.
    for j, h in enumerate(steps):
        lower, upper = INFERENCE_BOUNDS[j]
        plus, minus = theta.copy(), theta.copy()
        plus[j] += h
        minus[j] -= h
        # Prefer central differences when both perturbations satisfy bounds.
        if minus[j] >= lower and plus[j] <= upper:
            D[:, j] = (inference_mean_moments(plus, seeds) - inference_mean_moments(minus, seeds)) / (2*h)
            methods.append("central")
        elif plus[j] <= upper:
            D[:, j] = (inference_mean_moments(plus, seeds) - base) / h
            methods.append("forward")
        elif minus[j] >= lower:
            D[:, j] = (base - inference_mean_moments(minus, seeds)) / h
            methods.append("backward")
        else:
            raise ValueError("Finite-difference step is larger than the allowed parameter interval.")
    return D, methods


def standard_errors_from_jacobian(D, W_star, S=S_ESTIMATION):
    """
    Calculate parameter covariance and standard errors from moment derivatives.

    Parameters
    ----------
    D : ndarray, shape (6, 5)
        Numerical Jacobian of the simulated moments.
    W_star : ndarray, shape (6, 6)
        Inverse covariance of single-panel moment vectors from Module 7.
    S : int
        Number of independent simulated panels averaged in the SMM objective.
        This is not the number of covariance panels or derivative panels.

    Returns
    -------
    covariance_theta : ndarray, shape (5, 5)
        Parameter covariance using (1 + 1/S) * inverse(D.T @ W_star @ D).
    standard_errors : ndarray, shape (5,)
        Square roots of the parameter covariance diagonal.

    Raises
    ------
    ValueError
        If dimensions or S are invalid, derivatives lack full column rank,
        the information matrix is ill conditioned, or variances are nonpositive.

    Notes
    -----
    The panel-moment covariance already includes the finite-panel sampling
    scale, so there is no additional division by firms, periods, or covariance
    replications. Reliability still depends on derivative stability and the
    assumptions underlying the assignment's covariance formula. Parameter
    estimates on bounds require separate inference considerations.
    """
    if D.shape != (6, 5) or S < 1:
        raise ValueError("Need a 6-by-5 Jacobian and S >= 1.")
    # With Omega equal to the covariance of panel-level moments, it already
    # contains the finite-panel sampling scale. Do not divide by firms, T, or R again.
    # Check that the weighted moment derivatives identify five directions.
    whitened = np.linalg.cholesky(W_star).T @ D
    if np.linalg.matrix_rank(whitened) < 5:
        raise ValueError("Moment Jacobian has rank below 5: ordinary standard errors are not identified on this grid.")
    # Combine moment sensitivity with the second-stage weights.
    information = D.T @ W_star @ D
    information = (information + information.T) / 2
    if np.linalg.cond(information) > 1e12:
        raise ValueError("D' W D is too ill conditioned for reliable standard errors.")
    # Apply the simulation adjustment and invert the information matrix.
    covariance_theta = (1 + 1 / S) * np.linalg.solve(information, np.eye(5))
    covariance_theta = (covariance_theta + covariance_theta.T) / 2
    if np.any(np.diag(covariance_theta) <= 0):
        raise ValueError("Parameter covariance has nonpositive diagonal entries.")
    return covariance_theta, np.sqrt(np.diag(covariance_theta))

In [18]:
# Calculate matched moments at the Stage 1  and Stage 2 estimates
moments_stage1 = inference_mean_moments(
    theta_stage1, ESTIMATION_SEEDS
)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

stage1_moment_table = pd.DataFrame(
    {
        "Target": target_moments,
        "Stage 1": moments_stage1,
        "Difference": moments_stage1 - np.asarray(target_moments),
    },
    index=MOMENT_NAMES,
)
stage1_moment_table.index.name = "Moment"

# Formatted table for latex

display(stage1_moment_table.round(4))

stage1_moment_table.to_latex(
    "results/stage1_moments.tex",
    float_format="%.4f",
    column_format="lrrrrrr",
    escape=True,
)


#Construct W matrix for stage 2

# Construct the Stage 2 weighting matrix
replicated_moments, moment_covariance, W_star = (
    estimate_moment_covariance(theta_stage1)
)

# Estimate parameters using the inverse-covariance weights
stage2 = estimate_inference_stage(
    theta_stage1,
    W_star,
    "Efficient stage",
)
theta_stage2 = stage2["theta"]

print("Stage-2 estimates:",
      dict(zip(PARAMETER_NAMES, theta_stage2)))
print("Stage-2 converged:", stage2["converged"])
print(f"Stage-2 objective: {stage2['loss']:.6f}")

# Calculate moments at the Stage 2 estimates
moments_stage2 = inference_mean_moments(
    theta_stage2,
    ESTIMATION_SEEDS,
)

stage2_table = pd.DataFrame(
    {
        "Target": target_moments,
        "Stage 2": moments_stage2,
        "Difference": moments_stage2 - np.asarray(target_moments),
    },
    index=MOMENT_NAMES,
)
stage2_table.index.name = "Moment"

display(stage2_table.round(4))

stage2_table.to_latex(
    "results/stage2_moments.tex",
    float_format="%.4f",
    column_format="lrrr",
    escape=True,
)

NameError: name 'theta_stage1' is not defined

In [ ]:
#Standard Error Work


# jacobian, derivative_methods = numerical_moment_jacobian(theta_stage2, FD_STEPS)
jacobian_half, derivative_methods_half = numerical_moment_jacobian(theta_stage2, FD_STEPS / 2)
parameter_covariance, standard_errors = standard_errors_from_jacobian(jacobian, W_star, S_ESTIMATION)
parameter_covariance_half, standard_errors_half = standard_errors_from_jacobian(jacobian_half, W_star, S_ESTIMATION)
relative_derivative_change = (np.linalg.norm(jacobian - jacobian_half, axis=0)
                              / np.maximum(np.linalg.norm(jacobian_half, axis=0), 1e-12))
near_bound = np.minimum(theta_stage2 - INFERENCE_BOUNDS[:, 0],
                        INFERENCE_BOUNDS[:, 1] - theta_stage2) < 1e-4
stable_derivatives = bool(np.all(relative_derivative_change < 0.25))
inference_checks_pass = bool(stage1["converged"] and stage2["converged"]
                             and stable_derivatives and not np.any(near_bound))
standard_errors_table = pd.DataFrame({
    "Identity estimate": theta_stage1,
    "Efficient estimate": theta_stage2,
    "SE (h)": standard_errors,
    "SE (h/2)": standard_errors_half,
    "Derivative change": relative_derivative_change,
    "Difference method": derivative_methods,
    "At bound": near_bound,
}, index=PARAMETER_NAMES)
print(standard_errors_table.to_string(float_format=lambda x: f"{x:.4f}"))
print("Status:", "Numerical checks passed" if inference_checks_pass else "PROVISIONAL: review convergence, derivative sensitivity and boundary diagnostics")
print(f"S={S_ESTIMATION}; R={R_COVARIANCE}; simulation adjustment={1 + 1/S_ESTIMATION:.4f}")
print("Jacobian shape:", jacobian.shape)
print("Jacobian rank:", np.linalg.matrix_rank(jacobian))
print("Information matrix condition number:", np.linalg.cond(jacobian.T @ W_star @ jacobian))